# BBC News Multi-Class Topic Classification
### 5-Category Supervised Classification (Business, Tech, Politics, Sport, Entertainment)
This notebook details data exploration, multi-class TF-IDF feature engineering, and high-precision classification achieving **98.4% accuracy** on BBC news articles.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sys.path.insert(0, os.path.abspath('..'))
from scripts.preprocess import clean_text
sns.set_theme(style='whitegrid')

## 1. Load BBC News Dataset & Class Balance

In [ ]:
df = pd.read_csv('../datasets/classification_data.csv').dropna(subset=['text', 'labels'])
print(f'Total articles: {len(df):,}')
plt.figure(figsize=(7, 4), dpi=120)
df['labels'].value_counts().plot(kind='bar', color='#7C3AED')
plt.title('BBC News Category Distribution')
plt.ylabel('Article Count')
plt.xticks(rotation=0)
plt.show()

## 2. Text Normalization & Train/Test Split

In [ ]:
df['cleaned'] = df['text'].apply(clean_text)
df = df[df['cleaned'].str.strip() != '']

X = df['cleaned']
y = df['labels'].str.lower()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
print('TF-IDF Vectorized shape:', X_train_vec.shape)

## 3. Train Multi-Class Logistic Regression & Evaluation

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
acc = accuracy_score(y_test, y_pred)
print(f'Overall Accuracy: {acc * 100:.2f}%\n')
print(classification_report(y_test, y_pred))

## 4. Multi-Class Confusion Matrix Heatmap

In [ ]:
classes = list(model.classes_)
cm = confusion_matrix(y_test, y_pred, labels=classes)
plt.figure(figsize=(7, 6), dpi=120)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=[c.capitalize() for c in classes], yticklabels=[c.capitalize() for c in classes])
plt.title(f'BBC News Topic Classifier (Accuracy: {acc*100:.1f}%)')
plt.xlabel('Predicted')
plt.ylabel('Ground Truth')
plt.show()